# 03 — Silver y calidad
**Spec:** docs/specs/03_silver_calidad.md · **Requisitos:** R-08, R-09, R-10 · **Escribe:** `iol_silver.trades`, `iol_ops.quality_baselines`, `iol_ops.quality_events`

Silver es 1:1 con Bronze: ninguna fila se descarta, cada problema queda en `quality_flags` y `quality_status`.
Catálogo y argumentos de cada regla: `docs/quality_rules.md`. Los outputs no muestran IDs de cliente ni de operación.

In [0]:
dbutils.widgets.text("pipeline_run_id", "")
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.dropdown("demo_rerun", "true", ["true", "false"])

import json
import os
import sys
import uuid

sys.path.insert(0, os.path.abspath("../src"))
from iol_pipeline import config, io, silver

RUN_ID = dbutils.widgets.get("pipeline_run_id") or str(uuid.uuid4())
CATALOG = dbutils.widgets.get("catalog")
print("pipeline_run_id:", RUN_ID, "· rules_version:", config.RULES_VERSION)

io.ensure_schemas(spark, catalog=CATALOG)
bronze_table = io.table_name("bronze", "trades_raw", catalog=CATALOG)
silver_table = io.table_name("silver", "trades", catalog=CATALOG)
baselines_table = io.table_name("ops", "quality_baselines", catalog=CATALOG)
events_table = io.table_name("ops", "quality_events", catalog=CATALOG)

pipeline_run_id: 1d773c7a-5422-4fe5-96a6-a64397bc9037 · rules_version: rules_v1


## Corrida
`demo_rerun = true` la corre dos veces: la segunda tiene que dar 0 filas insertadas, 0 actualizadas y 0 eventos tocados. El orquestador la corre una vez.

In [0]:
first = silver.run_silver(spark, bronze_table, silver_table, baselines_table, events_table, RUN_ID)
print("1ra corrida:", first)
second = None
if dbutils.widgets.get("demo_rerun") == "true":
    second = silver.run_silver(spark, bronze_table, silver_table, baselines_table, events_table, RUN_ID)
    print("2da corrida:", second)

1ra corrida: {'bronze_rows': 100000, 'silver_rows': 100000, 'silver_inserted': 100000, 'silver_updated': 0, 'baselines_updated': 0, 'baselines_inserted': 1467, 'events_inserted': 10810, 'events_updated': 0}
2da corrida: {'bronze_rows': 100000, 'silver_rows': 100000, 'silver_inserted': 0, 'silver_updated': 0, 'baselines_updated': 0, 'baselines_inserted': 0, 'events_inserted': 0, 'events_updated': 0}


## 1. Reglas: filas afectadas
`example_record_hash` es un prefijo de `record_hash`: lleva a la fila de Bronze sin publicar el ID.

In [0]:
trades = spark.table(silver_table)
display(silver.rule_summary(trades))

rule_code,severity,rows,example_record_hash,pct_of_rows
DQ002_REQUIRED_NULL,CRITICAL,0,null,0.0
DQ003_TS_UNPARSEABLE,CRITICAL,0,null,0.0
DQ004_ID_CONFLICT,CRITICAL,0,null,0.0
DQ005_NUMERIC_UNPARSEABLE,CRITICAL,0,null,0.0
DQ006_NON_POSITIVE,ERROR,0,null,0.0
DQ007_DOMAIN_INVALID,ERROR,0,null,0.0
DQ008_MALFORMED_ROW,CRITICAL,0,null,0.0
DQ010_SATURDAY,WARNING,10255,000de33ca73f,10.255
DQ011_SUNDAY,ERROR,0,null,0.0
DQ012_MARKET_HOLIDAY,ERROR,0,null,0.0


## 2. Distribución de `quality_status`

In [0]:
display(silver.status_distribution(trades))

quality_status,rows,pct_of_rows
ERROR,8,0.008
OK,89290,89.29
WARNING,10702,10.702


## 3. Reconciliación Bronze contra Silver
Todas las diferencias tienen que dar 0.

In [0]:
check = silver.reconciliation(spark.table(bronze_table), trades, spark.table(events_table))
for key, value in check.items():
    print(f"{key:30} {value:,}")
reconciled = check["bronze_rows"] == check["silver_rows"] and not any(v for k, v in check.items() if k not in ("bronze_rows", "silver_rows"))
print("Reconciliación:", "OK" if reconciled else "CON DIFERENCIAS")

bronze_rows                    100,000
silver_rows                    100,000
bronze_not_in_silver           0
silver_not_in_bronze           0
not_ok_without_flag            0
severe_flags_without_event     0
active_events_without_flag     0
Reconciliación: OK


## 4. Ejemplos de `quality_events`

In [0]:
display(silver.event_examples(spark.table(events_table)))

rule_code,severity,details_json
DQ010_SATURDAY,WARNING,"{""trade_date_local"":""2026-01-03"",""effective_trade_date"":""2026-01-02""}"
DQ010_SATURDAY,WARNING,"{""trade_date_local"":""2026-01-03"",""effective_trade_date"":""2026-01-02""}"
DQ010_SATURDAY,WARNING,"{""trade_date_local"":""2026-01-03"",""effective_trade_date"":""2026-01-02""}"
DQ020_EXTREME_AMOUNT,WARNING,"{""gross_amount"":0.00200000,""currency_code"":""USD"",""asset_type_rule"":""BONO_SOBERANO"",""robust_z"":-4.013363573375316}"
DQ020_EXTREME_AMOUNT,WARNING,"{""gross_amount"":0.00240000,""currency_code"":""USD"",""asset_type_rule"":""BONO_SOBERANO"",""robust_z"":-3.9420898316611512}"
DQ020_EXTREME_AMOUNT,WARNING,"{""gross_amount"":0.39920000,""currency_code"":""ARS"",""asset_type_rule"":""OPCION"",""robust_z"":-4.869962158511843}"
DQ021_POSSIBLE_SCALE_ERROR,ERROR,"{""price"":0.01000000,""symbol"":""AL30"",""symbol_median"":876.3192,""price_ratio"":1.141136699960471E-5}"
DQ021_POSSIBLE_SCALE_ERROR,ERROR,"{""price"":0.01000000,""symbol"":""DNC7D"",""symbol_median"":1.0712,""price_ratio"":0.009335324869305453}"
DQ021_POSSIBLE_SCALE_ERROR,WARNING,"{""price"":0.01000000,""symbol"":""GFGC97054F"",""symbol_median"":2.01245,""price_ratio"":0.004969067554473404}"


In [0]:
idempotent = second is None or second["silver_inserted"] == second["silver_updated"] == second["events_inserted"] == second["events_updated"] == 0
ok = reconciled and idempotent
dbutils.notebook.exit(json.dumps({"status": "SUCCESS" if ok else "FAILED", "pipeline_run_id": RUN_ID, "rows_out": check["silver_rows"]}))